# Dock selected ligands against 5-HT2C (6QBH)

This notebook docks only the five selected hit compounds and the two named reference ligands listed in `data/raw/5ht2c_docking_ligands.csv`. It does not read random/scaffold libraries, predicted pIC50 values, or DRD2 docking rankings.

The receptor is the local `docking/prepared/6QBH/6bqh.mmcif` structure. The co-crystallized ligand component `E2J` defines the binding-site box and is excluded from the prepared receptor. Ligand poses and Vina scores are written under `docking/runs/5ht2c_6qbh/`; the score table is written to `results/tables/5ht2c_docking_results_6qbh.csv`. Vina scores are docking scores, not measured target affinity.

1. Global Setup and Project Information
Aim: To evaluate the potential off-target binding of our top 5 selected DRD2 antagonist candidates against the 5-HT2C serotonin receptor (PDB: 6QBH). Assessing off-target affinity is a critical step in drug discovery to minimize potential side effects and ensure target selectivity, alongside evaluating established reference ligands (Ritanserin and Risperidone).

Software and Packages Used:
Python (Version 3.x)
`vina`: AutoDock Vina Python bindings for computational molecular docking.
`rdkit`, `meeko`: For ligand 3D conformer generation, protonation, and PDBQT formatting.
`openmm`, `pdbfixer`, `biopython`: For automated structural repair of the 5-HT2C receptor (resolving missing atoms and residues).
`csv`, `pathlib`, `re`: For data parsing, dynamic path resolution, and string sanitization.

Global Settings & Parameters:
`BOX_PADDING_ANGSTROM = 8.0`: Expands the docking box radially from the native ligand (E2J) to allow sufficient conformational sampling.
`EXHAUSTIVENESS = 16`: Set to a high value to ensure rigorous sampling of the binding pocket.
`NUM_POSES = 9`: Maximum number of binding poses generated per ligand.
`RANDOM_SEED = 22`: Fixed seed for reproducible 3D conformer generation and Vina docking runs.

Required Input Files:
`data/raw/5ht2c_docking_ligands.csv`: Contains the 5 selected DRD2 hits and 2 reference ligands.
`docking/prepared/6QBH/6bqh.mmcif`: The raw 5-HT2C receptor structural file.

In [ ]:
from __future__ import annotations

import csv
import math
import re
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/raw/5ht2c_docking_ligands.csv").is_file() and (
            candidate / "docking/prepared/6QBH/6bqh.mmcif"
        ).is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run this notebook from the project directory."
    )


ROOT = find_project_root()
LIGANDS_CSV = ROOT / "data" / "raw" / "5ht2c_docking_ligands.csv"
RECEPTOR_CIF = ROOT / "docking" / "prepared" / "6QBH" / "6bqh.mmcif"
PREPARED_DIR = RECEPTOR_CIF.parent
RUN_DIR = ROOT / "docking" / "runs" / "5ht2c_6qbh"
RESULTS_CSV = ROOT / "results" / "tables" / "5ht2c_docking_results_6qbh.csv"

NATIVE_LIGAND_COMPONENT = "E2J"
NATIVE_LIGAND_CHAIN = "A"
BOX_PADDING_ANGSTROM = 8.0
EXHAUSTIVENESS = 16
NUM_POSES = 9
RANDOM_SEED = 22

print(f"Project: {ROOT}")
print(f"Input ligands: {LIGANDS_CSV}")
print(f"6QBH structure: {RECEPTOR_CIF}")


## Check docking tools

The local Linux/WSL docking environment needs RDKit, Biopython, PDBFixer, Meeko and AutoDock Vina.

In [ ]:
def require_dependencies() -> dict[str, str]:
    """
    Verify required computational environments and locate Meeko preparation binaries.
    
    Returns:
        dict[str, str]: Dictionary mapping Meeko tool names to their absolute paths.
    """
    if sys.platform == "win32":
        raise RuntimeError("Run the docking notebook in Linux/WSL or macOS, not native Windows.")

    try:
        from Bio.PDB.MMCIF2Dict import MMCIF2Dict  # noqa: F401
        from openmm.app import PDBFile  # noqa: F401
        from pdbfixer import PDBFixer  # noqa: F401
        from rdkit import Chem  # noqa: F401
        from rdkit.Chem import AllChem  # noqa: F401
        from vina import Vina  # noqa: F401
    except ImportError as exc:
        raise RuntimeError(
            "Missing a docking dependency. Activate the project's drd2-vina-py312 environment."
        ) from exc

    scripts_dir = Path(sysconfig.get_path("scripts"))
    tools: dict[str, str] = {}
    for name in ("mk_prepare_receptor.py", "mk_prepare_ligand.py"):
        executable = shutil.which(name)
        candidate = scripts_dir / name
        if executable is None and candidate.is_file():
            executable = str(candidate)
        if executable is None:
            raise FileNotFoundError(f"Meeko tool not found: {name}")
        tools[name] = executable
    return tools


tools = require_dependencies()
print("Docking dependencies and Meeko preparation tools are available.")


## Prepare the 6QBH receptor and native-ligand box

Only the 5-HT2C receptor residues are retained (author numbering 46–245 and 301–387); the inserted BRIL fusion, crystallographic ligand, lipids, PEG, and waters are excluded. The `E2J` heavy-atom coordinates are used to center and size the box with 8 Å padding.

2. Receptor Extraction and Structural Repair
Purpose: Isolate the precise biological binding unit of the 5-HT2C receptor and repair crystallographic defects to prepare an energetically stable target for Vina.

In [ ]:
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from pdbfixer import PDBFixer
from openmm.app import PDBFile


def cif_column(cif: dict, name: str) -> list[str]:
    """
    Safely extract a data column from a parsed mmCIF dictionary.
    
    Args:
        cif (dict): The parsed mmCIF data structure.
        name (str): The column key to extract.
        
    Returns:
        list[str]: Extracted data unified into a list format.
    """
    value = cif.get(name, [])
    return [value] if isinstance(value, str) else list(value)


def pdb_atom_line(serial: int, atom: dict[str, str], chain: str) -> str:
    atom_name = atom["atom"].strip()
    altloc = " " if atom["altloc"] in {".", "?", " "} else atom["altloc"]
    insertion = " " if atom["insertion"] in {".", "?", " "} else atom["insertion"]
    return (
        f"ATOM  {serial:5d} {atom_name:>4s}{altloc}{atom['resname']:>3s} {chain}"
        f"{int(atom['residue']):4d}{insertion}   "
        f"{float(atom['x']):8.3f}{float(atom['y']):8.3f}{float(atom['z']):8.3f}"
        f"{float(atom['occupancy']):6.2f}{float(atom['bfactor']):6.2f}"
        f"          {atom['element'].strip().upper():>2s}\n"
    )


def extract_receptor_and_box() -> tuple[Path, list[float], list[float]]:
    cif = MMCIF2Dict(str(RECEPTOR_CIF))
    names = {
        "record": "_atom_site.group_PDB",
        "element": "_atom_site.type_symbol",
        "atom": "_atom_site.label_atom_id",
        "altloc": "_atom_site.label_alt_id",
        "resname": "_atom_site.label_comp_id",
        "x": "_atom_site.Cartn_x",
        "y": "_atom_site.Cartn_y",
        "z": "_atom_site.Cartn_z",
        "occupancy": "_atom_site.occupancy",
        "bfactor": "_atom_site.B_iso_or_equiv",
        "residue": "_atom_site.auth_seq_id",
        "chain": "_atom_site.auth_asym_id",
        "insertion": "_atom_site.pdbx_PDB_ins_code",
        "model": "_atom_site.pdbx_PDB_model_num",
    }
    columns = {key: cif_column(cif, value) for key, value in names.items()}
    count = len(columns["record"])
    if not count or any(len(values) != count for values in columns.values()):
        raise ValueError(f"Incomplete atom-site data in {RECEPTOR_CIF}.")

    ligand_points: list[tuple[float, float, float]] = []
    receptor_atoms: dict[tuple[str, str, str], tuple[float, int, dict[str, str]]] = {}
    ranges = ((46, 245, "A"), (301, 387, "B"))
    for index in range(count):
        if columns["model"][index] not in {"1", ".", "?"}:
            continue
        component = columns["resname"][index]
        record = columns["record"][index]
        element = columns["element"][index].strip().upper()
        point = tuple(float(columns[axis][index]) for axis in ("x", "y", "z"))
        if (
            record == "HETATM"
            and component == NATIVE_LIGAND_COMPONENT
            and columns["chain"][index] == NATIVE_LIGAND_CHAIN
            and element not in {"H", "D"}
        ):
            ligand_points.append(point)
        if record != "ATOM" or columns["chain"][index] != "A":
            continue
        residue = int(columns["residue"][index])
        segment = next((chain for lower, upper, chain in ranges if lower <= residue <= upper), None)
        if segment is None or element in {"H", "D"}:
            continue
        atom = {key: columns[key][index] for key in names if key in {
            "element", "atom", "altloc", "resname", "x", "y", "z", "occupancy",
            "bfactor", "residue", "insertion"
        }}
        atom_key = (segment, str(residue), atom["atom"])
        occupancy = float(atom["occupancy"])
        previous = receptor_atoms.get(atom_key)
        if previous is None or occupancy > previous[0]:
            receptor_atoms[atom_key] = (occupancy, index, atom)

    if not ligand_points:
        raise ValueError(
            f"No heavy atoms for native ligand {NATIVE_LIGAND_COMPONENT} in chain "
            f"{NATIVE_LIGAND_CHAIN} of {RECEPTOR_CIF}."
        )
    if not receptor_atoms:
        raise ValueError("No 5-HT2C receptor atoms found in the expected 6QBH residue ranges.")

    receptor_pdb = PREPARED_DIR / "5ht2c_receptor_clean.pdb"
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    sorted_atoms = sorted(receptor_atoms.values(), key=lambda value: value[1])
    lines = ["REMARK 5-HT2C receptor from local 6QBH; BRIL fusion and heterogens excluded\n"]
    previous_chain = None
    serial = 1
    for _, _, atom in sorted_atoms:
        residue = int(atom["residue"])
        chain = next(chain for lower, upper, chain in ranges if lower <= residue <= upper)
        if previous_chain is not None and chain != previous_chain:
            lines.append(f"TER   {serial:5d}\n")
            serial += 1
        lines.append(pdb_atom_line(serial, atom, chain))
        serial += 1
        previous_chain = chain
    lines.extend([f"TER   {serial:5d}\n", "END\n"])
    receptor_pdb.write_text("".join(lines), encoding="ascii")

    minima = [min(point[axis] for point in ligand_points) for axis in range(3)]
    maxima = [max(point[axis] for point in ligand_points) for axis in range(3)]
    center = [(low + high) / 2 for low, high in zip(minima, maxima)]
    size = [high - low + 2 BOX_PADDING_ANGSTROM for low, high in zip(minima, maxima)]
    if any(value > 126 for value in size):
        raise ValueError(f"Calculated Vina box exceeds 126 Å: {size}")
    print(f"Selected {len(receptor_atoms)} receptor heavy atoms across 5-HT2C residues 46–245 and 301–387.")
    print(f"Found {len(ligand_points)} co-crystallized {NATIVE_LIGAND_COMPONENT} heavy atoms.")
    print(f"Box center: {[round(value, 3) for value in center]}; size: {[round(value, 3) for value in size]} Å")
    return receptor_pdb, center, size


receptor_pdb, box_center, box_size = extract_receptor_and_box()
repaired_receptor_pdb = PREPARED_DIR / "5ht2c_receptor_repaired.pdb"
fixer = PDBFixer(filename=str(receptor_pdb))
fixer.findMissingResidues()
fixer.missingResidues = {}
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
with repaired_receptor_pdb.open("w", encoding="ascii") as handle:
    PDBFile.writeFile(fixer.topology, fixer.positions, handle, keepIds=True)

receptor_pdbqt = PREPARED_DIR / "5ht2c_receptor.pdbqt"
subprocess.run([
    tools["mk_prepare_receptor.py"], "--read_pdb", str(repaired_receptor_pdb),
    "--write_pdbqt", str(receptor_pdbqt), "--box_center",
    *(f"{value:.3f}" for value in box_center), "--box_size",
    *(f"{value:.3f}" for value in box_size),
], check=True)
if not receptor_pdbqt.is_file() or receptor_pdbqt.stat().st_size == 0:
    raise RuntimeError(f"Meeko did not create a receptor PDBQT at {receptor_pdbqt}.")
print(f"Prepared 5-HT2C receptor: {receptor_pdbqt}")


## Load and prepare the seven named ligands

The input CSV is the sole ligand source: five selected hits and the two reference ligands. The notebook validates each row and does not query any prediction tables.

3. Target Ligand Preparation
Purpose: Convert the rigid 2D SMILES strings of our 5 candidates and 2 reference compounds into 3D, energetically minimized representations in PDBQT format.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem


with LIGANDS_CSV.open(newline="", encoding="utf-8") as handle:
    ligands = list(csv.DictReader(handle))
required_columns = {"selection_rank", "compound_id", "compound_name", "smiles", "role"}
if not ligands or not required_columns.issubset(ligands[0]):
    raise ValueError(f"Expected columns {sorted(required_columns)} in {LIGANDS_CSV}.")
if len(ligands) != 7:
    raise ValueError(f"Expected exactly 7 ligands (5 selected hits + 2 references), found {len(ligands)}.")
if sum(row["role"] == "selected_hit" for row in ligands) != 5:
    raise ValueError("Input must contain exactly five selected_hit rows.")
if {row["compound_name"] for row in ligands if row["role"] == "reference_ligand"} != {
    "Ritanserin", "Risperidone"
}:
    raise ValueError("Input must contain the Ritanserin and Risperidone reference ligands.")


def safe_name(text: str) -> str:
    slug = re.sub(r"[^A-Za-z0-9_-]+", "_", text).strip("_")
    if not slug:
        raise ValueError(f"Cannot create a safe file name from {text!r}.")
    return slug


RUN_DIR.mkdir(parents=True, exist_ok=True)
for ligand in ligands:
    molecule = Chem.MolFromSmiles(ligand["smiles"])
    if molecule is None:
        raise ValueError(f"Invalid SMILES for {ligand['compound_name']}: {ligand['smiles']}")
    name = safe_name(ligand["compound_id"])
    molecule.SetProp("_Name", ligand["compound_name"])
    molecule = Chem.AddHs(molecule)
    if AllChem.EmbedMolecule(molecule, randomSeed=RANDOM_SEED) != 0:
        raise ValueError(f"Could not generate 3D coordinates for {ligand['compound_name']}.")
    if AllChem.MMFFHasAllMoleculeParams(molecule):
        AllChem.MMFFOptimizeMolecule(molecule)
    sdf_path = RUN_DIR / name / f"{name}_3d.sdf"
    sdf_path.parent.mkdir(parents=True, exist_ok=True)
    writer = Chem.SDWriter(str(sdf_path))
    writer.write(molecule)
    writer.close()
    ligand["run_name"] = name
    ligand["sdf_path"] = sdf_path
    ligand["pdbqt_path"] = sdf_path.with_name(f"{name}.pdbqt")
    subprocess.run([
        tools["mk_prepare_ligand.py"], "-i", str(sdf_path), "-o", str(ligand["pdbqt_path"])
    ], check=True)
    if not ligand["pdbqt_path"].is_file() or ligand["pdbqt_path"].stat().st_size == 0:
        raise RuntimeError(f"Meeko did not prepare {ligand['compound_name']} as PDBQT.")
    print(f"Prepared {ligand['role']}: {ligand['compound_name']} ({ligand['compound_id']})")


## Dock only the listed ligands and save results

Each ligand is docked to the 6QBH receptor in the native `E2J` binding site. The score CSV records the ligand identity, role, selected-hit rank, and best Vina score; it contains no predicted activity or combined-ranking fields.

In [ ]:
from vina import Vina


def best_pose_score(poses_path: Path) -> float:
    for line in poses_path.read_text(encoding="utf-8").splitlines():
        if line.startswith("REMARK VINA RESULT:"):
            score = float(line.split()[3])
            if math.isfinite(score):
                return score
            raise ValueError(f"Non-finite Vina score in {poses_path}.")
    raise ValueError(f"No Vina score found in pose file {poses_path}.")


results: list[dict[str, str | float]] = []
for ligand in ligands:
    run_folder = RUN_DIR / ligand["run_name"]
    poses_path = run_folder / f"{ligand['run_name']}_poses.pdbqt"
    log_path = run_folder / "vina.log"
    vina = Vina(sf_name="vina", cpu=0, seed=RANDOM_SEED)
    vina.set_receptor(str(receptor_pdbqt))
    vina.set_ligand_from_file(str(ligand["pdbqt_path"]))
    vina.compute_vina_maps(center=box_center, box_size=box_size)
    with log_path.open("w", encoding="utf-8") as log_handle:
        from contextlib import redirect_stdout
        with redirect_stdout(log_handle):
            vina.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=NUM_POSES)
            vina.write_poses(str(poses_path), n_poses=NUM_POSES, overwrite=True)
    score = best_pose_score(poses_path)
    result = {
        "compound_id": ligand["compound_id"],
        "compound_name": ligand["compound_name"],
        "role": ligand["role"],
        "selection_rank": ligand["selection_rank"],
        "vina_score_kcal_mol": score,
        "receptor": "6QBH",
        "native_site_ligand": NATIVE_LIGAND_COMPONENT,
        "poses_file": str(poses_path.relative_to(ROOT)),
    }
    results.append(result)
    print(f"{ligand['compound_name']}: {score:.3f} kcal/mol ({poses_path})")

RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
fieldnames = [
    "compound_id", "compound_name", "role", "selection_rank",
    "vina_score_kcal_mol", "receptor", "native_site_ligand", "poses_file",
]
with RESULTS_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)
print(f"\nWrote {len(results)} docking results to {RESULTS_CSV}")
print("Vina scores are docking scores, not measured target affinity.")
